In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy import stats
from numpy.lib.stride_tricks import sliding_window_view
from IPython.display import display

In [2]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK/'input', TASK/'output'
DIAGNOSTICS = INPUT/'diagnostics'
CONFIG = json.loads((ROOT/'config.json').read_text())
Y, SEED = CONFIG['last_year'], CONFIG['seed']
MODELS = CONFIG['models']
EMPIRICAL = 'empirical_complete'
WINDOWS = {'full_observed':(0, Y), 'common_generated':(6, Y)}
DESCRIPTORS = ['active_fraction', 'longest_observed_zero_run', 'max_three_year_share',
               'normalized_total_variation', 'peak_age', 'log1p_trend_slope']

In [3]:
# source arrays follow the symlink chain through f_plot and e_diagnose
panel = pd.read_csv(DIAGNOSTICS/'simulation/fit/data/panel_complete.csv', index_col='dblp_id')
sources = {EMPIRICAL:panel.to_numpy(dtype=float).T}
sources.update({tag:np.load(DIAGNOSTICS/'simulation'/tag/'trajectories.npy', mmap_mode='r') for tag in MODELS})

In [4]:
def path_descriptors(array, start):
    n_years, n_careers = array.shape
    total = array.sum(axis=0)
    active_fraction = (array>0).mean(axis=0)
    run, longest = np.zeros(n_careers, dtype=int), np.zeros(n_careers, dtype=int)
    for yearly_output in array:
        run = np.where(yearly_output==0, run+1, 0)
        longest = np.maximum(longest, run)
    maximum_three = sliding_window_view(array, 3, axis=0).sum(axis=-1).max(axis=0)
    concentration = np.divide(maximum_three, total, out=np.full(n_careers, np.nan), where=total>0)
    # all-zero paths have no variation; concentration and peak age are undefined
    variation = np.divide(np.abs(np.diff(array, axis=0)).sum(axis=0), total,
                          out=np.zeros(n_careers), where=total>0)
    peak_age = np.where(total>0, start+array.argmax(axis=0), np.nan)
    centered_age = np.arange(n_years)-(n_years-1)/2
    slope = centered_age @ np.log1p(array)/(centered_age @ centered_age)
    return {'active_fraction':active_fraction, 'longest_observed_zero_run':longest,
            'max_three_year_share':concentration, 'normalized_total_variation':variation,
            'peak_age':peak_age, 'log1p_trend_slope':slope}
def descriptor_values(array):
    return {name:values[np.isfinite(values)] for name, values in array.items()}
def ecdf_knots(values):
    ordered = np.sort(values)
    x = np.unique(np.r_[ordered[0], np.quantile(ordered, np.linspace(0, 1, 401)), ordered[-1]])
    return pd.DataFrame({'x':x, 'cdf_left':np.searchsorted(ordered, x, side='left')/len(ordered),
                         'cdf':np.searchsorted(ordered, x, side='right')/len(ordered), 'n':len(ordered)})

In [5]:
# compare distributions of path features; no artificial one-to-one scholar matching
summary_rows, distance_rows, cdf_frames = [], [], []
for window, (start, end) in WINDOWS.items():
    reference = descriptor_values(path_descriptors(sources[EMPIRICAL][start:end+1], start))
    for source, array in sources.items():
        descriptors = descriptor_values(path_descriptors(array[start:end+1], start))
        for descriptor in DESCRIPTORS:
            values, observed = descriptors[descriptor], reference[descriptor]
            summary_rows.append({'source':source, 'window':window, 'start_age':start, 'end_age':end,
                                 'descriptor':descriptor, 'n_total':array.shape[1], 'n_defined':len(values),
                                 'mean':values.mean(), 'median':np.median(values), 'sd':values.std(ddof=0),
                                 'q05':np.quantile(values, .05), 'q95':np.quantile(values, .95),
                                 'includes_empirical_prefix':source=='ar6_6' and start<6})
            cdf_frames.append(ecdf_knots(values).assign(source=source, window=window, descriptor=descriptor))
            if source!=EMPIRICAL:
                distance_rows.append({'source':source, 'window':window, 'descriptor':descriptor,
                                      'n_empirical':len(observed), 'n_simulated':len(values),
                                      'empirical_mean':observed.mean(), 'simulated_mean':values.mean(),
                                      'empirical_median':np.median(observed), 'simulated_median':np.median(values),
                                      'ks_D':stats.ks_2samp(observed, values, method='asymp').statistic,
                                      'wasserstein':stats.wasserstein_distance(observed, values),
                                      'includes_empirical_prefix':source=='ar6_6' and start<6})
path_summary = pd.DataFrame(summary_rows)
path_distances = pd.DataFrame(distance_rows)
path_cdfs = pd.concat(cdf_frames, ignore_index=True)
path_summary.to_csv(OUTPUT/'path_summary.csv', index=False)
path_distances.to_csv(OUTPUT/'path_agreement.csv', index=False)
path_cdfs.to_csv(OUTPUT/'path_ecdf_knots.csv', index=False)
display(path_distances.loc[path_distances.window.eq('common_generated')])

,source,window,descriptor,n_empirical,n_simulated,empirical_mean,simulated_mean,empirical_median,simulated_median,ks_D,wasserstein,includes_empirical_prefix
36,arw4,common_generated,active_fraction,510,10000,0.878693,1.000000,0.933333,1.000000,0.545098,0.121307,False
37,arw4,common_generated,longest_observed_zero_run,510,10000,1.054902,0.000000,1.000000,0.000000,0.545098,1.054902,False
38,arw4,common_generated,max_three_year_share,510,10000,0.346021,0.364788,0.327497,0.354125,0.175782,0.022557,False
39,arw4,common_generated,normalized_total_variation,510,10000,0.622430,0.485447,0.565626,0.470533,0.258053,0.139706,False
40,arw4,common_generated,peak_age,510,10000,11.890196,12.161000,12.000000,12.000000,0.040647,0.332027,False
41,arw4,common_generated,log1p_trend_slope,510,10000,-0.010891,-0.008400,-0.011683,-0.009071,0.046657,0.005930,False
42,lognormal_grw,common_generated,active_fraction,510,10000,0.878693,0.880853,0.933333,1.000000,0.069898,0.010931,False
43,lognormal_grw,common_generated,longest_observed_zero_run,510,10000,1.054902,1.189800,1.000000,0.000000,0.085802,0.300831,False
44,lognormal_grw,common_generated,max_three_year_share,510,10000,0.346021,0.573705,0.327497,0.554724,0.688537,0.227736,False
45,lognormal_grw,common_generated,normalized_total_variation,510,10000,0.622430,0.715260,0.565626,0.676693,0.233337,0.098299,False


In [6]:
definitions = pd.DataFrame([
    {'descriptor':'active_fraction', 'definition':'fraction of years with positive output', 'caveat':'time-order invariant; contextual activity summary'},
    {'descriptor':'longest_observed_zero_run', 'definition':'longest consecutive run of zero-output years inside the window', 'caveat':'runs at window boundaries are truncated, not complete spell lengths'},
    {'descriptor':'max_three_year_share', 'definition':'largest sum over three consecutive years / total output in the window', 'caveat':'undefined for all-zero paths; descriptive concentration, not an excess-over-shuffling test'},
    {'descriptor':'normalized_total_variation', 'definition':'sum of absolute adjacent-year changes / total output in the window', 'caveat':'defined as zero for all-zero paths; measures fluctuation relative to output'},
    {'descriptor':'peak_age', 'definition':'career age of maximum output within the window, earliest year on ties', 'caveat':'undefined for all-zero paths; first-maximum convention affects flat careers'},
    {'descriptor':'log1p_trend_slope', 'definition':'OLS slope of log(1 + annual output) against career age within each path', 'caveat':'descriptive direction of within-career change; not a causal effect'}
])
definitions.to_csv(OUTPUT/'path_metric_definitions.csv', index=False)
display(definitions)

,descriptor,definition,caveat
0,active_fraction,fraction of years with positive output,time-order invariant; contextual activity summary
1,longest_observed_zero_run,longest consecutive run of zero-output years i...,"runs at window boundaries are truncated, not c..."
2,max_three_year_share,largest sum over three consecutive years / tot...,undefined for all-zero paths; descriptive conc...
3,normalized_total_variation,sum of absolute adjacent-year changes / total ...,defined as zero for all-zero paths; measures f...
4,peak_age,career age of maximum output within the window...,undefined for all-zero paths; first-maximum co...
5,log1p_trend_slope,OLS slope of log(1 + annual output) against ca...,descriptive direction of within-career change;...


In [7]:
# metric-specific options; keep population-curve agreement separate from path structure
statement_rows = []
for row in path_distances.loc[path_distances.window.eq('common_generated')].itertuples():
    statement_rows.append({'source':row.source, 'window':row.window, 'descriptor':row.descriptor,
                           'text':f'For {row.descriptor.replace("_", " ")} over years 6--{Y}, the empirical median was {row.empirical_median:.3g} and the simulated median was {row.simulated_median:.3g}; the two-sample KS distance was {row.ks_D:.3f}.'})
path_statements = pd.DataFrame(statement_rows)
path_statements.to_csv(OUTPUT/'path_discussion_options.csv', index=False)
display(path_statements)

,source,window,descriptor,text
0,arw4,common_generated,active_fraction,"For active fraction over years 6--20, the empi..."
1,arw4,common_generated,longest_observed_zero_run,For longest observed zero run over years 6--20...
2,arw4,common_generated,max_three_year_share,"For max three year share over years 6--20, the..."
3,arw4,common_generated,normalized_total_variation,For normalized total variation over years 6--2...
4,arw4,common_generated,peak_age,"For peak age over years 6--20, the empirical m..."
5,arw4,common_generated,log1p_trend_slope,"For log1p trend slope over years 6--20, the em..."
6,lognormal_grw,common_generated,active_fraction,"For active fraction over years 6--20, the empi..."
7,lognormal_grw,common_generated,longest_observed_zero_run,For longest observed zero run over years 6--20...
8,lognormal_grw,common_generated,max_three_year_share,"For max three year share over years 6--20, the..."
9,lognormal_grw,common_generated,normalized_total_variation,For normalized total variation over years 6--2...
